# OmniVoice FastAPI + Cloudflare Quick Tunnel

Notebook này chạy API clone giọng OmniVoice trên Google Colab và public tạm thời qua Cloudflare Quick Tunnel. Trước khi chạy, chọn **Runtime → Change runtime type → GPU**.

> Quick Tunnel sinh hostname `trycloudflare.com` ngẫu nhiên, hết hiệu lực khi tunnel dừng và bất cứ ai biết URL + API key đều có thể gọi. Chỉ dùng cho demo/test; không đưa key bí mật hoặc API production vào notebook.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'Hãy chuyển runtime sang GPU trước khi tiếp tục.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Không ép phiên bản torch để giữ CUDA/PyTorch tương thích với runtime Colab.
!pip -q install -U omnivoice fastapi 'uvicorn[standard]' python-multipart soundfile httpx gradio

In [ ]:
import os
import secrets
import secrets

# Giữ key này riêng tư. Chạy lại cell để tạo key mới trước khi mở tunnel mới.
API_KEY = secrets.token_urlsafe(32)
os.environ['OMNIVOICE_API_KEY'] = API_KEY
print('API key (lưu lại trước khi chia sẻ URL):', API_KEY)

In [ ]:
%%writefile /content/omnivoice_api.py
import asyncio
import io
import os
import tempfile
from contextlib import asynccontextmanager
from pathlib import Path

import gradio as gr
import soundfile as sf
import torch
from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.concurrency import run_in_threadpool
from fastapi.responses import StreamingResponse
from omnivoice import OmniVoice

MODEL_ID = 'k2-fsa/OmniVoice'
SAMPLE_RATE = 24000
API_KEY = os.environ.get('OMNIVOICE_API_KEY')
if not API_KEY:
    raise RuntimeError('OMNIVOICE_API_KEY is required')

model = None
generation_lock = asyncio.Lock()  # One GPU generation at a time on the Colab runtime.

@asynccontextmanager
async def lifespan(app: FastAPI):
    global model
    model = OmniVoice.from_pretrained(
        MODEL_ID, device_map='cuda:0', dtype=torch.float16
    )
    yield
    model = None
    torch.cuda.empty_cache()

app = FastAPI(title='OmniVoice TTS API', version='0.1.0', lifespan=lifespan)

def write_wav(audio):
    waveform = audio[0]
    if hasattr(waveform, 'detach'):
        waveform = waveform.detach().cpu().numpy()
    output = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    output.close()
    sf.write(output.name, waveform, SAMPLE_RATE)
    return output.name

def ui_generate(text, ref_audio, ref_text):
    if not text or not text.strip():
        raise gr.Error('Hãy nhập nội dung cần đọc.')
    if not ref_audio:
        raise gr.Error('Hãy upload audio tham chiếu.')
    if not ref_text or not ref_text.strip():
        raise gr.Error('Hãy nhập transcript chính xác của audio tham chiếu.')
    audio = model.generate(text=text.strip(), ref_audio=ref_audio, ref_text=ref_text.strip())
    return write_wav(audio)

with gr.Blocks(title='OmniVoice TTS') as ui:
    gr.Markdown('# OmniVoice — Vietnamese voice cloning')
    with gr.Row():
        with gr.Column():
            ref_audio_ui = gr.Audio(label='Reference audio (3–10 giây, một người nói)', type='filepath')
            ref_text_ui = gr.Textbox(label='Transcript của reference audio', lines=3)
        with gr.Column():
            text_ui = gr.Textbox(label='Nội dung cần đọc', lines=5, placeholder='Xin chào! Đây là OmniVoice.')
            generate_ui = gr.Button('Generate', variant='primary')
            output_ui = gr.Audio(label='Kết quả', type='filepath')
    generate_ui.click(ui_generate, [text_ui, ref_audio_ui, ref_text_ui], output_ui)

ui.queue(default_concurrency_limit=1)
# UI dùng Basic Auth; username là admin, password là OMNIVOICE_API_KEY.
app = gr.mount_gradio_app(app, ui, path='/ui', auth=('admin', API_KEY), show_error=True)

def require_api_key(x_api_key: str = Header(...)):
    if not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail='Invalid API key')

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_ID}

@app.post('/tts', dependencies=[Depends(require_api_key)])
async def tts(
    text: str = Form(..., min_length=1, max_length=2000),
    ref_text: str = Form(..., min_length=1, max_length=2000),
    ref_audio: UploadFile = File(...),
):
    if not ref_audio.content_type or not ref_audio.content_type.startswith('audio/'):
        raise HTTPException(status_code=415, detail='ref_audio must be an audio file')
    suffix = Path(ref_audio.filename or 'reference.wav').suffix or '.wav'
    payload = await ref_audio.read()
    if len(payload) > 25 * 1024 * 1024:
        raise HTTPException(status_code=413, detail='Reference audio must be <= 25 MB')

    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as temp:
        temp.write(payload)
        reference_path = temp.name
    try:
        async with generation_lock:
            audio = await run_in_threadpool(
                model.generate, text=text, ref_audio=reference_path, ref_text=ref_text
            )
        output_path = write_wav(audio)
        buffer = io.BytesIO(Path(output_path).read_bytes())
        Path(output_path).unlink(missing_ok=True)
        return StreamingResponse(
            buffer, media_type='audio/wav',
            headers={'Content-Disposition': 'attachment; filename=omnivoice.wav'},
        )
    finally:
        Path(reference_path).unlink(missing_ok=True)


In [ ]:
# Uvicorn nạp model một lần; log giữ tại /content/omnivoice_api.log.
!pkill -f 'uvicorn omnivoice_api:app' || true
!nohup uvicorn omnivoice_api:app --app-dir /content --host 127.0.0.1 --port 8000 > /content/omnivoice_api.log 2>&1 & echo $! > /content/omnivoice_api.pid

import time
import httpx

for _ in range(90):
    try:
        response = httpx.get('http://127.0.0.1:8000/health', timeout=3)
        response.raise_for_status()
        print(response.json())
        break
    except Exception:
        time.sleep(2)
else:
    !tail -100 /content/omnivoice_api.log
    raise RuntimeError('API did not start; inspect the log above.')

In [ ]:
# Optional: upload một audio tham chiếu và test API nội bộ trước khi public.
from google.colab import files
uploaded = files.upload()
assert uploaded, 'Hãy upload audio tham chiếu.'
reference_file = next(iter(uploaded))
REFERENCE_TRANSCRIPT = 'Thay bằng transcript chính xác của audio tham chiếu.'
TEXT_TO_SPEAK = 'Xin chào! Đây là một yêu cầu thử nghiệm qua API OmniVoice.'
assert 'Thay bằng' not in REFERENCE_TRANSCRIPT

with open(reference_file, 'rb') as handle:
    response = httpx.post(
        'http://127.0.0.1:8000/tts',
        headers={'X-API-Key': API_KEY},
        data={'text': TEXT_TO_SPEAK, 'ref_text': REFERENCE_TRANSCRIPT},
        files={'ref_audio': (reference_file, handle, 'audio/wav')},
        timeout=300,
    )
response.raise_for_status()
open('/content/local_test.wav', 'wb').write(response.content)
print('Saved /content/local_test.wav')

In [ ]:
# Cài cloudflared bản Linux chính thức của Cloudflare.
!curl --fail --location --output /tmp/cloudflared.deb "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-$(dpkg --print-architecture).deb"
!dpkg -i /tmp/cloudflared.deb
!cloudflared --version

In [ ]:
import re
import subprocess
import time
from pathlib import Path

# Quick Tunnel không cần Cloudflare account/domain. URL chỉ sống khi process này còn chạy.
Path('/content/cloudflared.log').unlink(missing_ok=True)
tunnel_log = open('/content/cloudflared.log', 'w')
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(60):
    time.sleep(1)
    log_text = Path('/content/cloudflared.log').read_text(errors='replace')
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', log_text)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    print(log_text)
    raise RuntimeError('Cloudflare Quick Tunnel did not start.')

print('Public API:', public_url)
print('Test UI:', public_url + '/ui', '(Basic Auth: admin / API key)')
print('Swagger docs:', public_url + '/docs')
print('Health check:', public_url + '/health')

## Gọi API qua URL public

Mở **`{public_url}/ui`** để test qua giao diện upload audio / nhập transcript / generate. Browser sẽ hỏi Basic Auth: username là `admin`, password là `API_KEY` được in ở đầu notebook. UI này tiện thử thủ công; endpoint API vẫn dùng URL bên dưới.

Endpoint: `POST {public_url}/tts` — gửi multipart form gồm `text`, `ref_text`, và file `ref_audio`; bắt buộc header `X-API-Key`. `ref_text` phải khớp chính xác audio tham chiếu.

Ví dụ Python (chạy sau cell tunnel):

```python
import requests

with open('reference.wav', 'rb') as audio:
    response = requests.post(
        f'{public_url}/tts',
        headers={'X-API-Key': API_KEY},
        data={
            'text': 'Xin chào, đây là API OmniVoice.',
            'ref_text': 'Bản chép lời chính xác của reference.wav.',
        },
        files={'ref_audio': ('reference.wav', audio, 'audio/wav')},
        timeout=300,
    )
response.raise_for_status()
open('output.wav', 'wb').write(response.content)
```

Quick Tunnel phù hợp demo/development: Cloudflare nêu rõ URL là tạm thời, không có uptime guarantee, và giới hạn 200 request đang xử lý đồng thời. Với hostname cố định hoặc production, tạo **named/remotely-managed Cloudflare Tunnel** trong tài khoản Cloudflare, dùng tunnel token/hostname của bạn và bổ sung rate-limit, auth mạnh hơn cùng logging.

In [ ]:
# Chạy cell này khi kết thúc demo để ngừng public access và giải phóng GPU.
if 'tunnel_process' in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    tunnel_process.wait(timeout=15)
!pkill -f 'uvicorn omnivoice_api:app' || true
print('Tunnel and API stopped.')